In [1]:
import os
os.listdir('/data')

['dim_artists.csv',
 'dim_tracks.csv',
 'dim_users.csv',
 'generate_events.py',
 'clean_data.py',
 'test_queries.py',
 'fact_streams.csv',
 'profile_data.py',
 'venv',
 'dataset.csv',
 'track_genres.csv',
 'load_to_db.py']

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SonicFlow") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Spark is running!")

Spark version: 3.5.0
Spark is running!


In [3]:
# Load fact_streams with Spark
df_streams = spark.read.csv("/data/fact_streams.csv", header=True, inferSchema=True)

# How pandas shows info vs how Spark shows info
print("Schema (Spark's way of showing column types):")
df_streams.printSchema()

print(f"Total rows: {df_streams.count()}")
print(f"Partitions: {df_streams.rdd.getNumPartitions()}")

Schema (Spark's way of showing column types):
root
 |-- stream_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- track_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- listen_duration_ms: integer (nullable = true)
 |-- skipped: boolean (nullable = true)
 |-- shuffle: boolean (nullable = true)
 |-- device: string (nullable = true)

Total rows: 500000
Partitions: 10


In [4]:
# This does NOT run yet - Spark just remembers what you asked
filtered = df_streams.filter(df_streams.skipped == True)
with_duration = filtered.withColumn("listen_sec", filtered.listen_duration_ms / 1000)
result = with_duration.select("stream_id", "track_id", "listen_sec")

print("Nothing has executed yet! Spark just built a plan.")
print(f"Plan: {result.explain()}")

Nothing has executed yet! Spark just built a plan.
== Physical Plan ==
*(1) Project [stream_id#17, track_id#19, (cast(listen_duration_ms#21 as double) / 1000.0) AS listen_sec#55]
+- *(1) Filter (isnotnull(skipped#22) AND skipped#22)
   +- FileScan csv [stream_id#17,track_id#19,listen_duration_ms#21,skipped#22] Batched: false, DataFilters: [isnotnull(skipped#22), skipped#22], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/fact_streams.csv], PartitionFilters: [], PushedFilters: [IsNotNull(skipped), EqualTo(skipped,true)], ReadSchema: struct<stream_id:int,track_id:string,listen_duration_ms:int,skipped:boolean>


Plan: None


In [5]:
# THIS triggers execution - .show() is an "action"
result.show(5)

+---------+--------------------+----------+
|stream_id|            track_id|listen_sec|
+---------+--------------------+----------+
|        9|6qBJhWis1EGlcDvC7...|    40.801|
|       13|1XT5kxg6Tk0ukCO2v...|    40.553|
|       20|0qdQUeKVyevrbKhAo...|    66.098|
|       38|2GHnPQ8mAukedPzVZ...|    52.395|
|       42|6MaRBjJJtf3IiWHRV...|    24.001|
+---------+--------------------+----------+
only showing top 5 rows



In [6]:
# Load tracks too
df_tracks = spark.read.csv("/data/dim_tracks.csv", header=True, inferSchema=True)

# Top 10 most streamed tracks - same query we did in SQL!
from pyspark.sql.functions import count, desc

top_tracks = df_streams \
    .groupBy("track_id") \
    .agg(count("*").alias("stream_count")) \
    .join(df_tracks, "track_id") \
    .select("track_name", "artists", "stream_count") \
    .orderBy(desc("stream_count")) \
    .limit(10)

top_tracks.show(truncate=False)

+----------------------------------------------+----------------------------------------+------------+
|track_name                                    |artists                                 |stream_count|
+----------------------------------------------+----------------------------------------+------------+
|OUT OUT (feat. Charli XCX & Saweetie)         |Joel Corry;Jax Jones;Charli XCX;Saweetie|27          |
|Heaven Takes You Home (feat. Connie Constance)|Swedish House Mafia;Connie Constance    |26          |
|Streets                                       |Doja Cat                                |26          |
|I Wanna Be Yours                              |Arctic Monkeys                          |26          |
|Attention                                     |Charlie Puth                            |26          |
|Balader                                       |Soolking;Niska                          |25          |
|La Bachata                                    |Manuel Turizo            

In [7]:
# Run a groupBy + join so we have something interesting to look at in Spark UI
from pyspark.sql.functions import count, avg, desc

streams_by_country = df_streams \
    .join(
        spark.read.csv("/data/dim_users.csv", header=True, inferSchema=True),
        "user_id"
    ) \
    .groupBy("country") \
    .agg(
        count("*").alias("total_streams"),
        avg("listen_duration_ms").alias("avg_listen_ms")
    ) \
    .orderBy(desc("total_streams"))

streams_by_country.show()

+---------+-------------+------------------+
|  country|total_streams|     avg_listen_ms|
+---------+-------------+------------------+
|       US|       153372|129849.88184283963|
|    India|        69665|129828.73488839447|
|       UK|        60972|130273.77706160204|
|   Brazil|        52929| 129488.1713616354|
|  Germany|        39638|130675.91636813158|
|    Japan|        35177|129112.88026267164|
|   Canada|        24527|131153.58869816936|
|   Mexico|        24242|130964.25827076974|
|Australia|        20377|  129981.389017029|
|   France|        19101|130449.52918695356|
+---------+-------------+------------------+



In [8]:
# Let's see the FULL execution plan for our country query
streams_by_country.explain(True)

== Parsed Logical Plan ==
'Sort ['total_streams DESC NULLS LAST], true
+- Aggregate [country#214], [country#214, count(1) AS total_streams#252L, avg(listen_duration_ms#21) AS avg_listen_ms#254]
   +- Project [user_id#18, stream_id#17, track_id#19, timestamp#20, listen_duration_ms#21, skipped#22, shuffle#23, device#24, username#213, country#214, plan#215, age_group#216, signup_date#217]
      +- Join Inner, (user_id#18 = user_id#212)
         :- Relation [stream_id#17,user_id#18,track_id#19,timestamp#20,listen_duration_ms#21,skipped#22,shuffle#23,device#24] csv
         +- Relation [user_id#212,username#213,country#214,plan#215,age_group#216,signup_date#217] csv

== Analyzed Logical Plan ==
country: string, total_streams: bigint, avg_listen_ms: double
Sort [total_streams#252L DESC NULLS LAST], true
+- Aggregate [country#214], [country#214, count(1) AS total_streams#252L, avg(listen_duration_ms#21) AS avg_listen_ms#254]
   +- Project [user_id#18, stream_id#17, track_id#19, timestamp#20, 

In [9]:
# CSV vs Parquet - this is a HUGE concept in data engineering
# First, save our streams as Parquet
df_streams.write.mode("overwrite").parquet("/data/streams_parquet")

# Now read both and compare
import time

# Time CSV read + count
start = time.time()
spark.read.csv("/data/fact_streams.csv", header=True, inferSchema=True).count()
csv_time = time.time() - start

# Time Parquet read + count
start = time.time()
spark.read.parquet("/data/streams_parquet").count()
parquet_time = time.time() - start

print(f"CSV read + count:     {csv_time:.2f} seconds")
print(f"Parquet read + count: {parquet_time:.2f} seconds")
print(f"Parquet is {csv_time/parquet_time:.1f}x faster")

CSV read + count:     0.36 seconds
Parquet read + count: 0.15 seconds
Parquet is 2.3x faster


In [10]:
import os

def folder_size(path):
    total = 0
    for f in os.listdir(path):
        fp = os.path.join(path, f)
        if os.path.isfile(fp):
            total += os.path.getsize(fp)
    return total

csv_size = os.path.getsize("/data/fact_streams.csv")
parquet_size = folder_size("/data/streams_parquet")

print(f"CSV size:     {csv_size / 1024 / 1024:.1f} MB")
print(f"Parquet size: {parquet_size / 1024 / 1024:.1f} MB")
print(f"Parquet is {csv_size / parquet_size:.1f}x smaller")

CSV size:     38.4 MB
Parquet size: 17.6 MB
Parquet is 2.2x smaller


In [11]:
# Save streams partitioned by date (year/month)
from pyspark.sql.functions import year, month

df_streams \
    .withColumn("year", year("timestamp")) \
    .withColumn("month", month("timestamp")) \
    .write.mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/data/streams_partitioned")

# Let's see what this created
for root, dirs, files in os.walk("/data/streams_partitioned"):
    level = root.replace("/data/streams_partitioned", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for d in sorted(dirs):
            pass  # will show on next iteration

streams_partitioned/
  year=2024/
    month=1/
    month=6/
    month=3/
    month=4/
    month=5/
    month=2/


In [12]:
import time

# Read the partitioned data
df_partitioned = spark.read.parquet("/data/streams_partitioned")

# Query for just March - Spark only reads the month=3 folder
start = time.time()
march_count = df_partitioned.filter("month = 3").count()
partition_time = time.time() - start

# Same query on non-partitioned parquet - Spark reads everything
df_flat = spark.read.parquet("/data/streams_parquet")
start = time.time()
march_count2 = df_flat.filter(month("timestamp") == 3).count()
flat_time = time.time() - start

print(f"March streams: {march_count}")
print(f"Partitioned query:     {partition_time:.3f} seconds")
print(f"Non-partitioned query: {flat_time:.3f} seconds")
print(f"Partitioned is {flat_time/partition_time:.1f}x faster")

March streams: 85715
Partitioned query:     0.149 seconds
Non-partitioned query: 0.262 seconds
Partitioned is 1.8x faster


In [13]:
# Let's compare two ways to do the same join

df_users = spark.read.csv("/data/dim_users.csv", header=True, inferSchema=True)

# Method 1: Regular join (Spark auto-picks broadcast since users is small)
from pyspark.sql.functions import broadcast

result1 = df_streams.join(df_users, "user_id").groupBy("country").count()
result1.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[country#502], functions=[count(1)])
   +- Exchange hashpartitioning(country#502, 200), ENSURE_REQUIREMENTS, [plan_id=705]
      +- HashAggregate(keys=[country#502], functions=[partial_count(1)])
         +- Project [country#502]
            +- BroadcastHashJoin [user_id#18], [user_id#500], Inner, BuildRight, false
               :- Filter isnotnull(user_id#18)
               :  +- FileScan csv [user_id#18] Batched: false, DataFilters: [isnotnull(user_id#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/fact_streams.csv], PartitionFilters: [], PushedFilters: [IsNotNull(user_id)], ReadSchema: struct<user_id:int>
               +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=700]
                  +- Filter isnotnull(user_id#500)
                     +- FileScan csv [user_id#500,country#502] Batched: false, DataFilters: [isnotnull(use

In [14]:
# Method 2: Force a sort-merge join (what happens with two BIG tables)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # disable broadcast

result2 = df_streams.join(df_users, "user_id").groupBy("country").count()
result2.explain()

# Turn broadcast back on
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[country#502], functions=[count(1)])
   +- Exchange hashpartitioning(country#502, 200), ENSURE_REQUIREMENTS, [plan_id=751]
      +- HashAggregate(keys=[country#502], functions=[partial_count(1)])
         +- Project [country#502]
            +- SortMergeJoin [user_id#18], [user_id#500], Inner
               :- Sort [user_id#18 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(user_id#18, 200), ENSURE_REQUIREMENTS, [plan_id=743]
               :     +- Filter isnotnull(user_id#18)
               :        +- FileScan csv [user_id#18] Batched: false, DataFilters: [isnotnull(user_id#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/fact_streams.csv], PartitionFilters: [], PushedFilters: [IsNotNull(user_id)], ReadSchema: struct<user_id:int>
               +- Sort [user_id#500 ASC NULLS FIRST], false, 0
                  +- Exchange hashpartitioning(user_id#500, 200), ENSURE